# 2. Feature Binning

This notebook bins three numerical continuous features into categorical cohorts:
1. **`customer_age`** $\to$ `customer_age_binned` (`Youth`, `Young-Adult`, `Middle-Aged`, `Senior`)
2. **`transaction_hour`** $\to$ `transaction_hour_binned` (`Night`, `Morning`, `Afternoon`, `Evening`)
3. **`distance_to_merchant`** $\to$ `distance_to_merchant_binned` (`Close`, `Moderate`, `Far`)

## 1. Imports & Setup

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports successful.')

Imports successful.


## 2. Load Processed Data

In [2]:
INPUT_PATH = '../../dataset/processed/credit_card_fraud_outliers_handled.csv'

df = pd.read_csv(INPUT_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Loaded: 1,296,675 rows x 16 columns


,distance_to_merchant,customer_age,transaction_hour,day_of_week,is_weekend,location_mismatch,velocity_last_24h,foreign_transaction,amount,merchant_category,city_population,gender,is_fraud,amount_log,velocity_last_24h_log,city_population_log
0,127.6062,32,12,1,0,1,0,0,7.2700,misc_net,1645,F,0,2.1126,0.0000,7.4061
1,110.3089,32,8,2,0,1,1,0,52.9400,gas_transport,1645,F,0,3.9879,0.6931,7.4061
2,21.7873,32,8,2,0,0,2,0,82.0800,gas_transport,1645,F,0,4.4198,1.0986,7.4061
3,87.2042,32,12,2,0,1,3,0,34.7900,kids_pets,1645,F,0,3.5777,1.3863,7.4061
4,74.2130,32,13,2,0,0,3,0,27.1800,home,1645,F,0,3.3386,1.3863,7.4061


## 3. Apply Feature Binning to 3 Columns

In [3]:
def map_age_cohort(age):
    if pd.isna(age):
        return np.nan
    elif age < 30:
        return 'Youth'
    elif age < 45:
        return 'Young-Adult'
    elif age < 65:
        return 'Middle-Aged'
    else:
        return 'Senior'

def map_hour_cohort(hour):
    if pd.isna(hour):
        return np.nan
    elif hour >= 22 or hour < 6:
        return 'Night'
    elif hour < 12:
        return 'Morning'
    elif hour < 17:
        return 'Afternoon'
    else:
        return 'Evening'

def map_distance_cohort(dist):
    if pd.isna(dist):
        return np.nan
    elif dist < 10:
        return 'Close'
    elif dist < 80:
        return 'Moderate'
    else:
        return 'Far'

df_binned = df.copy()

# 1. Age binning
df_binned['customer_age_binned'] = df_binned['customer_age'].apply(map_age_cohort)
df_binned.drop(columns=['customer_age'], inplace=True)

# 2. Hour binning
df_binned['transaction_hour_binned'] = df_binned['transaction_hour'].apply(map_hour_cohort)
df_binned.drop(columns=['transaction_hour'], inplace=True)

# 3. Distance binning
df_binned['distance_to_merchant_binned'] = df_binned['distance_to_merchant'].apply(map_distance_cohort)
df_binned.drop(columns=['distance_to_merchant'], inplace=True)

print("Binned 'customer_age', 'transaction_hour', and 'distance_to_merchant' into categorical groups.")

Binned 'customer_age', 'transaction_hour', and 'distance_to_merchant' into categorical groups.


## 4. Verify the Transformations

In [4]:
print('Dtype summary after binning:')
print(df_binned.dtypes.value_counts())

print()
for col in ['customer_age_binned', 'transaction_hour_binned', 'distance_to_merchant_binned']:
    print(f"Value counts for {col}:")
    print(df_binned[col].value_counts(dropna=False))
    print()

Dtype summary after binning:
int64      7
object     5
float64    4
Name: count, dtype: int64

Value counts for customer_age_binned:
customer_age_binned
Young-Adult    437895
Middle-Aged    420767
Youth          243473
Senior         194540
Name: count, dtype: int64

Value counts for transaction_hour_binned:
transaction_hour_binned
Night        388916
Evening      327640
Afternoon    326573
Morning      253546
Name: count, dtype: int64

Value counts for distance_to_merchant_binned:
distance_to_merchant_binned
Moderate    667075
Far         619034
Close        10566
Name: count, dtype: int64



## 5. Save Outputs

In [5]:
OUTPUT_PATH = '../../dataset/processed/credit_card_fraud_binned.csv'

output_dir = os.path.dirname(OUTPUT_PATH)
os.makedirs(output_dir, exist_ok=True)
df_binned.to_csv(OUTPUT_PATH, index=False)

print(f'Saved binned dataset to: {OUTPUT_PATH}')
print(f'Rows: {df_binned.shape[0]:,} | Columns: {df_binned.shape[1]:,}')

Saved binned dataset to: ../../dataset/processed/credit_card_fraud_binned.csv
Rows: 1,296,675 | Columns: 16
